# **MODEL 1: Multimodal Integration - Late Fusion (EfficientNetB1 + Metadata MLP)**

**Objective:**
Inclusion of **clinical metadata**
- **Age**
- **Sex**
- **Atomic location** of the lesion
<br>

**Methodology:**
| Variable | Strategy | NA Treatment |
|----------|-----------|--------------------|
| Age | Normalization (min-max) | Imputation by median |
| Sex | One-hot encoding | Category "unknown" |
| Location | One-hot encoding | Category "unknown" |

## 1. Imports

In [1]:
import tensorflow as tf
#prevent TF from grabbing all GPU memory at once
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [2]:
import os, sys, json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, label_binarize
from sklearn.utils.class_weight import compute_class_weight
import keras.applications
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from keras.applications.efficientnet import preprocess_input

if os.getcwd().endswith('models'):
    os.chdir('..')

from utils.utils_model import *
from utils.utils_augmentation import *
from utils.utils_preproc import *

In [3]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
print("TF built with CUDA:", tf.test.is_built_with_cuda())
print("GPU available to TF:", tf.test.is_gpu_available())  # deprecated but still works

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF built with CUDA: True
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
GPU available to TF: True


## 2. Data Configuration and loading

In [4]:
if os.getcwd().endswith('models'):
    os.chdir('..')

In [5]:
BASE_PATH = "./data"
TRAIN_CSV = os.path.join(BASE_PATH, "augmented_metadata.csv")
VAL_CSV = os.path.join(BASE_PATH, "val_split.csv")
TEST_CSV = os.path.join(BASE_PATH, "test_split.csv")

MODEL_SAVE_PATH = "./models/best_model_m1.keras"


In [6]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

## 3. Metadata preprocessing

Clean and encode the clinical features (Age, Sex, Localization) into a fixed-length vector before the split.

In [7]:
def preprocess_metadata(df, scaler=None, encoder=None, is_training=True):
    """Clean and encode clinical features into a fixed-length vector."""
    df = df.copy()

    # Impute missing values
    df['age'] = df['age'].fillna(df['age'].median())
    df['sex'] = df['sex'].fillna('unknown')
    df['localization'] = df['localization'].fillna('unknown')

    # Scale age
    if is_training:
        scaler = MinMaxScaler()
        age_scaled = scaler.fit_transform(df[['age']])
    else:
        age_scaled = scaler.transform(df[['age']])

    # One-hot encode categorical columns
    cat_cols = ['sex', 'localization']
    if is_training:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded_cats = encoder.fit_transform(df[cat_cols])
    else:
        encoded_cats = encoder.transform(df[cat_cols])

    meta_vectors = np.hstack([age_scaled, encoded_cats])
    return meta_vectors, scaler, encoder

In [8]:
train_meta, scaler, encoder = preprocess_metadata(train_df, is_training=True)
val_meta, _, _ = preprocess_metadata(val_df, scaler, encoder, is_training=False)
test_meta, _, _ = preprocess_metadata(test_df, scaler, encoder, is_training=False)

In [9]:
# Verification
META_DIM = train_meta.shape[1]
print(f"Metadata vector dimension: {META_DIM}")

Metadata vector dimension: 19


## 4. Multimodal Data Pipeline

In [10]:
def load_multimodal_item(path, meta, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])   # <--- FIX: B0 uses 224x224!
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return {"image_input": img, "meta_input": meta}, label

In [11]:
def create_ds(df, meta, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        df['image_path'].values,
        meta,
        df['dx_encoded'].values.astype(np.int32)
    )).map(load_multimodal_item, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)

    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

In [12]:
train_ds = create_ds(train_df, train_meta)
val_ds = create_ds(val_df, val_meta)
test_ds = create_ds(test_df, test_meta)

## 5. Architecture Late Fusion: EfficientNetB1 + Metadata MLP 

This builds the two branches and concatenates them into the final classification head. Establishing correct connections between EfficientNet and the MLP

In [13]:
import os

print("Current working directory:", os.getcwd())

if os.path.exists("checkpoints"):
    print("\nFiles in checkpoints folder:")
    for file in os.listdir("checkpoints"):
        print(f" - {file}")
else:
    print("\nERROR: I cannot see a 'checkpoints' folder from here!")

Current working directory: c:\Users\faust\Desktop\deep-learning-project

Files in checkpoints folder:
 - history_phase1.json
 - history_phase2.json
 - model_b_best.weights.h5
 - model_b_best_ENv2s.weights.h5
 - model_b_phase1.weights.h5
 - model_CUSTOM_best.weights.h5
 - model_CUSTOM_phase1.weights.h5
 - model_DN_phase1.weights.h5
 - model_DN_phase2.weights.h5
 - model_EN_phase1.weights.h5
 - model_EN_phase2.weights.h5
 - model_MN_phase1.weights.h5
 - model_MN_phase2.weights.h5
 - model_V2B1_phase1.weights.h5
 - model_V2B1_phase2.weights.h5


In [ ]:
#v1 import

import tensorflow as tf
from tensorflow.keras import mixed_precision

NUM_CLASSES = 7

# Match the policy the checkpoint was saved with
mixed_precision.set_global_policy('mixed_float16')

# 1. Create the base using tf.keras (to perfectly match your multimodal setup)
_base_tmp = tf.keras.applications.EfficientNetB0(
    include_top=False, weights=None, input_shape=(224, 224, 3), pooling=None
)

# 2. Build a dummy model around it
_inp = tf.keras.Input(shape=(224, 224, 3))
_x = _base_tmp(_inp, training=False)
_x = tf.keras.layers.GlobalAveragePooling2D()(_x)
_out = tf.keras.layers.Dense(NUM_CLASSES)(_x)

_loader = tf.keras.Model(_inp, _out)

# 3. Load by name and ignore mismatches in the head!
_loader.load_weights("checkpoints/model_EN_phase2.weights.h5", by_name=True, skip_mismatch=True)

# 4. THE FIX: Transfer weights by matching exact names, not list indices!
transferred_count = 0
for dst_layer in base_model.layers:
    try:
        # Find the layer with the exact same name in our loaded model
        src_layer = _base_tmp.get_layer(dst_layer.name)
        dst_layer.set_weights(src_layer.get_weights())
        transferred_count += 1
    except ValueError:
        # If a layer doesn't exist by name, skip it
        pass

# Clean up memory
del _loader, _base_tmp

# Restore default policy for the rest of the multimodal notebook
mixed_precision.set_global_policy('float32')
print(f"Backbone weights transferred successfully! ({transferred_count} layers matched)")

In [16]:
import h5py

with h5py.File("checkpoints/model_EN_phase2.weights.h5", "r") as f:
    print("--- TOP LEVEL LAYERS IN .H5 FILE ---")
    keys = list(f.keys())
    print(keys)
    
    print("\n--- FINDING THE BACKBONE ---")
    for key in keys:
        if 'efficientnet' in key.lower() or 'model' in key.lower() or 'base' in key.lower():
            print(f"Backbone found: {key}")
            if isinstance(f[key], h5py.Group):
                inner_keys = list(f[key].keys())
                print(f"Number of layers inside backbone: {len(inner_keys)}")
                print(f"First 5 layers: {inner_keys[:5]}")

--- TOP LEVEL LAYERS IN .H5 FILE ---
['batch_normalization_1', 'dense_2', 'dense_3', 'dropout_2', 'dropout_3', 'efficientnetb0', 'global_average_pooling2d_1', 'input_4', 'top_level_model_weights']

--- FINDING THE BACKBONE ---
Backbone found: efficientnetb0
Number of layers inside backbone: 131
First 5 layers: ['block1a_bn', 'block1a_dwconv', 'block1a_project_bn', 'block1a_project_conv', 'block1a_se_expand']
Backbone found: top_level_model_weights
Number of layers inside backbone: 0
First 5 layers: []


In [19]:
import h5py
import tensorflow as tf
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

print("1. Building Multimodal base_model...")
# We explicitly define the model here so it always exists!
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, weights=None, input_shape=(224, 224, 3)
)

transferred_count = 0
print("2. Bypassing Keras loader... Manually extracting weights via h5py...")

# Open the .h5 file like a standard folder
with h5py.File("checkpoints/model_EN_phase2.weights.h5", "r") as f:
    # Dive directly into the pristine backbone, ignoring the corrupted top-level dense layers
    backbone_group = f['efficientnetb0']
    
    # Iterate through your Multimodal tf.keras base_model
    for dst_layer in base_model.layers:
        layer_name = dst_layer.name
        
        # If the layer has learned weights in the file, extract them
        if layer_name in backbone_group:
            layer_group = backbone_group[layer_name]
            
            # Keras tracks the internal arrays using a 'weight_names' attribute
            if 'weight_names' in layer_group.attrs:
                weight_names = layer_group.attrs['weight_names']
                # Decode bytes to strings
                weight_names = [n.decode('utf-8') if isinstance(n, bytes) else n for n in weight_names]
                
                weight_arrays = []
                for w_name in weight_names:
                    try:
                        # Extract the raw numpy array from the h5py group
                        if w_name in layer_group:
                            weight_arrays.append(layer_group[w_name][:])
                        elif layer_name in layer_group and w_name in layer_group[layer_name]:
                            weight_arrays.append(layer_group[layer_name][w_name][:])
                    except KeyError:
                        pass
                
                # If we found the exact right number of arrays, inject them!
                if len(weight_arrays) > 0 and len(weight_arrays) == len(dst_layer.weights):
                    try:
                        dst_layer.set_weights(weight_arrays)
                        transferred_count += 1
                    except ValueError:
                        pass # Silently skip any lingering shape mismatches

mixed_precision.set_global_policy('float32')
print(f"3. Surgical extraction complete! {transferred_count} layers perfectly matched and transferred.")

1. Building Multimodal base_model...
2. Bypassing Keras loader... Manually extracting weights via h5py...
3. Surgical extraction complete! 0 layers perfectly matched and transferred.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# 1. FIX THE BUG: Calculate class weights for the Multimodal notebook
classes = np.array(sorted(train_df["dx_encoded"].unique()))
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["dx_encoded"])
class_weights_dict = {i: w for i, w in enumerate(weights)}
print(f"Class weights calculated: {class_weights_dict}\n")

In [ ]:
# Phase 1: freeze the pretrained base entirely
base_model.trainable = False

image_input = layers.Input(shape=(224, 224, 3), name="image_input")
x = base_model(image_input, training=False)   # training=False keeps BN in inference mode while frozen
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)   # reduced to 128 to match meta branch
image_features = layers.Dropout(0.4)(x)       # dropout after dense

In [ ]:
# Metadata Stream (MLP)
meta_input = layers.Input(shape=(META_DIM,), name="meta_input")
y = layers.Dense(64, activation='relu')(meta_input)
y = layers.Dropout(0.3)(y)                    # dropout in MLP
y = layers.Dense(128, activation='relu')(y)   # raised to 128 to match image branch
meta_features = layers.Dropout(0.3)(y)        # dropout after final MLP dense

In [ ]:
# Concatenation (The Fusion Point)
combined = layers.Concatenate()([image_features, meta_features])  # 256-d (128 + 128)
combined = layers.Dense(128, activation='relu')(combined)          # optional fusion FC
combined = layers.Dropout(0.3)(combined)
final_output = layers.Dense(7, activation='softmax')(combined)

model = models.Model(inputs=[image_input, meta_input], outputs=final_output)

### Training

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', BalancedAccuracy(N_CLASSES)]
)

In [ ]:
model.summary()

## Phase 1 — Train classification head only (base frozen)

EarlyStopping and ModelCheckpoint ensure we save the best epoch and stop before overfitting. ReduceLROnPlateau decays LR when val_loss plateaus.

In [ ]:
MODEL_SAVE_PATH = "./models/best_model_m1.weights.h5"

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=5, 
        restore_best_weights=True, 
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH, 
        monitor='val_loss', 
        save_best_only=True, 
        save_weights_only=True, # <--- THIS IS THE FIX!
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=3, 
        min_lr=1e-6, 
        verbose=1
    )
]

In [ ]:
# 5. TRAIN PHASE 1
print("Phase 1: Training fusion head and MLP (base frozen)...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_phase1,
    class_weight=class_weights_dict, 
    verbose=1
)

## Phase 2 — Fine-tune: unfreeze base and train end-to-end

Unfreeze the entire EfficientNetB1 and re-compile with amuch smaller learning rate to avoid destroying the pretrained weights.

In [ ]:
base_model.trainable = True

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),   # 100x smaller LR
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy']
)

In [ ]:
callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
]

In [ ]:
print("Phase 2: fine-tuning (base unfrozen)")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_phase2,
    class_weight=class_weights_dict,   # FIX 5
    verbose=1
)

## Evaluation on Test Set

classification_report gives per-class precision/recall/F1,which is the correct metric for imbalanced multiclass classification.

In [ ]:
# Load best checkpoint before evaluating
model.load_weights(MODEL_SAVE_PATH)

print("Test set evaluation")
test_loss, test_acc, test_auc = model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc:.4f}")
print(f"Test AUC  : {test_auc:.4f}")

In [ ]:
# Per-class metrics
y_true = test_df['dx_encoded'].values.astype(np.int32)

In [ ]:
# Collect predictions across all batches
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
# Decode label names
label_map = {v: k for k, v in
             dict(enumerate(sorted(train_df['dx'].unique()))).items()} \
            if 'dx' in train_df.columns else None

target_names = [label_map[i] for i in sorted(label_map.keys())] \
               if label_map else [str(i) for i in range(7)]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))